<a href="https://colab.research.google.com/github/josedanielisidororeyes/Deep_Learning_For_Finance/blob/main/Multilayer_Perceptron_NVIDIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Loading libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
# Downloading data for Nvidia
df = yf.download(["NVDA"], start = '2015-01-01', end  =  '2025-01-01')["Close"].reset_index()
df.columns  =  ["Date", "Close"]
df.columns.name  = None
df.set_index("Date", inplace  =  True)


# Creating returns, dropping first observation and implementing forward fill for missing prices
df["Ret"] =  df['Close'].pct_change()* 100
df =  df.iloc[1:].ffill().drop(columns  =  "Close")
df.head()

In [ ]:
# Computing cumulative returns
df['Cumulative Return'] =  (1 +  df["Ret"] / 100).cumprod() -1

# Cumulative returns for NVIDA
plt.figure(figsize =  (9, 6))
df["Cumulative Return"].plot()
plt.title("Cumulative Return of NVIDIA")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering using cumulative returns

df["Ret30_i"] =  df['Ret'].rolling(25).apply(lambda x: 100 * (np.prod(1 + x / 100) - 1))
df["Ret60_i"] =  df["Ret"].rolling(60).apply(lambda x: 100 * (np.prod(1 + x / 100) - 1))
df["Ret90_i"] =  df["Ret"].rolling(90).apply(lambda x: 100 * (np.prod(1 + x / 100) - 1))
df["Ret120_i"] =  df["Ret"].rolling(120).apply(lambda x: 100 * (np.prod(1 + x / 100) -1))
df["Ret240_i"] =  df["Ret"].rolling(240).apply(lambda x: 100 * (np.prod(1 + x / 100) - 1))
df =  df.dropna()
df.tail(5)

In [ ]:
# Defining the target | Increases or Decreases in Returns in the next 60 days
df["Ret60"] = df["Ret60_i"].shift(-60)
df["Output"] = df["Ret60"] > 0
df["Output"] = df["Output"].astype(int)
del df["Ret60"]
del df["Cumulative Return"]
df = df.dropna()
df.tail(10)

In [ ]:
# Train-Test Samples and Scaling
ts =  int(0.2 *  len(df))
split_time  =  len(df) - ts # Start of test sample
test_time  =  df.index[split_time:] # Extract date index for test period
Ret_vector  =  df["Ret"][split_time:].values
df.tail()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xdf = df.drop(columns=["Output"])
ydf = df["Output"]

X = Xdf.astype("float32")
y = ydf.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ts, shuffle=False
)

X_train =  X_train.iloc[:-60]
y_train  =  y_train.iloc[:-60]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train.values
y_test = y_test.values


In [ ]:
# Model and training
import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(5)

act_fun  = "relu"
hp_units  =  25
hp_units_2 =  15
hp_units_3 =  10
n_dropout  =  0.2

model  =  tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(units  =  hp_units, activation  =  act_fun))
model.add(tf.keras.layers.Dropout(n_dropout))
model.add(tf.keras.layers.Dense(units  =  hp_units_2, activation  = act_fun))
model.add(tf.keras.layers.Dropout(n_dropout))
model.add(tf.keras.layers.Dense(units  =  hp_units_3, activation  =  act_fun))
model.add(tf.keras.layers.Dropout(n_dropout))
model.add(tf.keras.layers.Dense(units  =  1, activation  =  "sigmoid"))

hp_lr  =  1e-5

adam  =  tf.keras.optimizers.Adam(learning_rate = hp_lr) # Adam optimizer
model.compile(optimizer =  adam, loss = 'binary_crossentropy', metrics  = ['accuracy'])

In [ ]:
# Validation and callbacks  | eartly stopping
es =  tf.keras.callbacks.EarlyStopping(
    monitor  = "val_accuracy",
    mode =  "max",
    verbose  =  1,
    patience =  20,
    restore_best_weights =  True,
)

In [ ]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)
total = len(y_train)

weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)

class_weight = {0: float(weight_for_0), 1: float(weight_for_1)}

In [ ]:
# Model training
history  =  model.fit(
    X_train,
    y_train,
    validation_split = 0.2,
    epochs  =  1000,
    batch_size  =  32,
    verbose  =  2,
    callbacks =  [es],
    class_weight = class_weight
    )

In [ ]:
model.summary()

In [ ]:
# Model evaluation
import seaborn as sns
from sklearn import metrics

y_prob  =  model.predict(X_test)
y_pred =  np.where(y_prob > 0.5, 1, 0)

acc = model.evaluate(X_test, y_test)
print("Model accuracy in test:", acc)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
cm = metrics.confusion_matrix(y_test, y_pred[:, 0])
disp =  ConfusionMatrixDisplay(cm, display_labels = ["Down", "Up"])
disp.plot()

In [ ]:
# Backtesting trading strategy
df_predictions  =  pd.DataFrame(
    {
        "Date": test_time,
        "Pred": y_pred.flatten() if y_pred.ndim > 1 else y_pred,
        "Ret": Ret_vector.flatten() if Ret_vector.ndim > 1 else Ret_vector
    }
)
df_predictions.tail()

In [ ]:
df =  df_predictions
df["Ret"] =  df["Ret"] / 100

df["Positions"] =  np.where(df["Pred"]> 0.5, 1, -1)
df["Strat_ret"] =  df["Positions"].shift(1) *  df["Ret"]
df["Positions_L"] =  df["Positions"].shift(1)
df["Positions_L"][df["Positions_L"] ==  -1]= 0
df["Strat_ret_L"] =  df["Positions_L"] *  df["Ret"]
df["CumRet"] =  df["Strat_ret"].expanding().apply(lambda x: np.prod(1 + x) -1)
df["CumRet_L"] =  df["Strat_ret_L"].expanding().apply(lambda x: np.prod(1 + x) -1)
df["bhRet"] =  df["Ret"].expanding().apply(lambda x: np.prod(1 + x) - 1)

fig =  plt.figure(figsize = (12, 6))
ax=  plt.gca()
df.plot(x =  "Date", y = 'bhRet', label = "Buy&Hold", ax  =  ax)
df.plot(x = "Date", y = "CumRet_L", label  = "Strategy Only Long", ax  =  ax)
df.plot(x  = "Date", y = "CumRet", label  = "Strategy Long and Short", ax  =  ax)
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.grid()
plt.show()